In [6]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv'

try:
    df = pd.read_csv(url)
except:
    url = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'
    df = pd.read_csv(url)

df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


In [3]:
print("Q1 Mode for industry:", df['industry'].mode()[0])

Q1 Mode for industry: retail


In [4]:
numerical = ['interaction_count', 'lead_score', 'number_of_courses_viewed', 'annual_income']
corr_matrix = df[numerical].corr()
print("Q2 Correlation Matrix:")
print(corr_matrix)

Q2 Correlation Matrix:
                          interaction_count  lead_score  \
interaction_count                  1.000000    0.009888   
lead_score                         0.009888    1.000000   
number_of_courses_viewed          -0.023565   -0.004879   
annual_income                      0.048618    0.005334   

                          number_of_courses_viewed  annual_income  
interaction_count                        -0.023565       0.048618  
lead_score                               -0.004879       0.005334  
number_of_courses_viewed                  1.000000       0.031551  
annual_income                             0.031551       1.000000  


In [7]:
from sklearn.metrics import mutual_info_score

categorical = ['industry', 'location', 'lead_source', 'employment_status']

df_cat = df[categorical].fillna('missing')

def calculate_mi(series):
    return mutual_info_score(series, df.converted)

mi_scores = df_cat.apply(calculate_mi)
print("Q3 Mutual Information Scores:")
print(mi_scores.sort_values(ascending=False))

Q3 Mutual Information Scores:
lead_source          0.026574
employment_status    0.011070
industry             0.007267
location             0.001427
dtype: float64


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import accuracy_score

numerical = ['interaction_count', 'lead_score', 'number_of_courses_viewed', 'annual_income']
categorical = ['industry', 'location', 'lead_source', 'employment_status']
features = categorical + numerical

# Split data
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values

# Fill missing values
df_train_filled = df_train[features].fillna(0)
df_val_filled = df_val[features].fillna(0)

# Encoding & Training
train_dicts = df_train_filled.to_dict(orient='records')
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val_filled.to_dict(orient='records')
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=10, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
acc = accuracy_score(y_val, y_pred)

print(f"Q4 Accuracy: {round(acc, 2)}")

Q4 Accuracy: 0.7


In [9]:
options = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

train_dicts = df_train_filled[features].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)
val_dicts = df_val_filled[features].to_dict(orient='records')
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=10, max_iter=1000, random_state=42)
model.fit(X_train, y_train)
base_score = accuracy_score(y_val, model.predict(X_val))

for c in options:
    subset = [f for f in features if f != c]
    
    dv_sub = DictVectorizer(sparse=False)
    X_train_sub = dv_sub.fit_transform(df_train_filled[subset].to_dict(orient='records'))
    X_val_sub = dv_sub.transform(df_val_filled[subset].to_dict(orient='records'))
    
    model_sub = LogisticRegression(solver='liblinear', C=10, max_iter=1000, random_state=42)
    model_sub.fit(X_train_sub, y_train)
    
    score = accuracy_score(y_val, model_sub.predict(X_val_sub))
    diff = abs(base_score - score)
    print(f"Bila {c:<25} | Accuracy: {score:.4f} | Tofauti: {diff:.4f}")

Bila lead_source               | Accuracy: 0.7031 | Tofauti: 0.0034
Bila number_of_courses_viewed  | Accuracy: 0.5563 | Tofauti: 0.1433
Bila interaction_count         | Accuracy: 0.5563 | Tofauti: 0.1433


In [10]:
c_values = [0.000001, 0.00001, 0.0001, 0.001]

train_dicts = df_train_filled[features].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)
val_dicts = df_val_filled[features].to_dict(orient='records')
X_val = dv.transform(val_dicts)

for C in c_values:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_val)
    score = accuracy_score(y_val, y_pred)
    print(f"C={C:<10} | Accuracy: {round(score, 3)}")

C=1e-06      | Accuracy: 0.556
C=1e-05      | Accuracy: 0.556
C=0.0001     | Accuracy: 0.556
C=0.001      | Accuracy: 0.563
